In [1]:
from pathlib import Path
import os
import random
import json
import pickle

import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

/home/jamyoung/miniconda3/envs/yolov3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ROOT = Path("../preprocess/preprocessing")
META_PATH = ROOT / "metadata.csv"

OUTPUT_DIR = Path("outputs_effnet_hbp_rf_minimal")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

classes = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
label2idx = {c: i for i, c in enumerate(classes)}
idx2label = {i: c for c, i in label2idx.items()}

seed = 42
model_name = "tf_efficientnetv2_m"

image_size = 300
batch_size = 8
num_workers = 0

out_indices = (2, 3, 4)
proj_dim = 64
pca_dim = 512
n_estimators = 300

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(seed)

print("device:", device)
print("model:", model_name)

device: cuda
model: tf_efficientnetv2_m


In [4]:
available = timm.list_models("*efficientnetv2*")
[m for m in available if "m" in m.lower()][:20]

['efficientnetv2_m', 'efficientnetv2_rw_m', 'tf_efficientnetv2_m']

In [5]:
df = pd.read_csv(META_PATH)

print(df.head())
print(df.columns)
print(df.shape)
print(df["dx"].value_counts())

required_cols = ["lesion_id", "image_id", "dx", "dx_type", "age", "sex", "localization", "dataset"]
for col in required_cols:
    assert col in df.columns, f"Missing column: {col}"

def get_image_path(row):
    cls = row["dx"]
    image_id = row["image_id"]
    path = ROOT / cls / "enhanced" / f"{image_id}.jpg"
    if path.exists():
        return str(path)
    return None

df["image_path"] = df.apply(get_image_path, axis=1)

missing = df["image_path"].isna().sum()
print("missing images:", missing)

if missing > 0:
    df[df["image_path"].isna()][["image_id", "dx", "image_path"]].to_csv(
        OUTPUT_DIR / "missing_images.csv",
        index=False
    )
    raise FileNotFoundError(f"{missing} images missing. Check missing_images.csv")

df["label"] = df["dx"].map(label2idx).astype(int)

print(df[["image_id", "dx", "label", "image_path"]].head())
print(df["label"].value_counts().sort_index())

     lesion_id      image_id   dx dx_type   age   sex localization  \
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp   
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp   
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp   
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp   
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear   

        dataset  
0  vidir_modern  
1  vidir_modern  
2  vidir_modern  
3  vidir_modern  
4  vidir_modern  
Index(['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization',
       'dataset'],
      dtype='object')
(10015, 8)
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64
missing images: 0
       image_id   dx  label                                         image_path
0  ISIC_0027419  bkl      2  ../preprocess/preprocessing/bkl/enhanced/ISIC_...
1  ISIC_0025030  bkl      2  ../prepr

In [6]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["dx"],
    random_state=seed
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("train:", len(train_df))
print(train_df["dx"].value_counts())

print("\nval:", len(val_df))
print(val_df["dx"].value_counts())

train_df.to_csv(OUTPUT_DIR / "train_split.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "val_split.csv", index=False)

train: 9013
dx
nv       6034
mel      1002
bkl       989
bcc       463
akiec     294
vasc      128
df        103
Name: count, dtype: int64

val: 1002
dx
nv       671
mel      111
bkl      110
bcc       51
akiec     33
vasc      14
df        12
Name: count, dtype: int64


In [7]:
train_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.03
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class HAMImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = int(row["label"])
        image_id = row["image_id"]

        return image, label, image_id

In [8]:
train_dataset = HAMImageDataset(train_df, transform=train_tfms)
val_dataset = HAMImageDataset(val_df, transform=val_tfms)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,       # 提特征时不需要 shuffle，方便对齐 image_id
    num_workers=num_workers,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

batch = next(iter(train_loader))
images, labels, image_ids = batch

print(images.shape)
print(labels.shape)
print(image_ids[:3])

torch.Size([8, 3, 300, 300])
torch.Size([8])
['ISIC_0026337', 'ISIC_0028005', 'ISIC_0029667']


In [9]:
feature_test_model = timm.create_model(
    model_name,
    pretrained=True,
    features_only=True,
    out_indices=out_indices
).to(device)

feature_test_model.eval()

with torch.no_grad():
    feats = feature_test_model(images.to(device))

for i, f in enumerate(feats):
    print(f"feature {i}: shape = {f.shape}")
    
channels = feature_test_model.feature_info.channels()
print("channels:", channels)

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


feature 0: shape = torch.Size([8, 80, 38, 38])
feature 1: shape = torch.Size([8, 176, 19, 19])
feature 2: shape = torch.Size([8, 512, 10, 10])
channels: [80, 176, 512]


In [10]:
def bilinear_pooling(f1, f2):
    """
    f1, f2: [B, D, H, W]
    return: [B, D*D]
    """
    B, D, H, W = f1.shape

    x = f1.view(B, D, H * W)
    y = f2.view(B, D, H * W)

    bilinear = torch.bmm(x, y.transpose(1, 2)) / (H * W)
    bilinear = bilinear.view(B, D * D)

    return bilinear


def normalize_bilinear_features(x, eps=1e-8):
    """
    Common normalization for bilinear features:
    signed square-root + L2 normalization.
    """
    x = torch.sign(x) * torch.sqrt(torch.abs(x) + eps)
    x = F.normalize(x, p=2, dim=1)
    return x

In [11]:
class HBPFeatureExtractor(nn.Module):
    def __init__(
        self,
        model_name="tf_efficientnetv2_m",
        out_indices=(2, 3, 4),
        proj_dim=64,
        pretrained=True
    ):
        super().__init__()

        self.feature_extractor = timm.create_model(
            model_name,
            pretrained=pretrained,
            features_only=True,
            out_indices=out_indices
        )

        channels = self.feature_extractor.feature_info.channels()

        self.projections = nn.ModuleList([
            nn.Conv2d(c, proj_dim, kernel_size=1)
            for c in channels
        ])

        self.proj_dim = proj_dim

    def forward(self, x):
        feats = self.feature_extractor(x)

        # 使用最后一层 feature map 的空间尺寸作为统一大小
        target_size = feats[-1].shape[-2:]

        projected = []

        for feat, proj in zip(feats, self.projections):
            z = proj(feat)
            z = F.interpolate(
                z,
                size=target_size,
                mode="bilinear",
                align_corners=False
            )
            projected.append(z)

        f1, f2, f3 = projected

        b12 = bilinear_pooling(f1, f2)
        b23 = bilinear_pooling(f2, f3)
        b13 = bilinear_pooling(f1, f3)

        hbp = torch.cat([b12, b23, b13], dim=1)
        hbp = normalize_bilinear_features(hbp)

        return hbp

In [12]:
hbp_model = HBPFeatureExtractor(
    model_name=model_name,
    out_indices=out_indices,
    proj_dim=proj_dim,
    pretrained=True
).to(device)

hbp_model.eval()

with torch.no_grad():
    hbp_feats = hbp_model(images.to(device))

print("HBP feature shape:", hbp_feats.shape)
print("Expected dim:", 3 * proj_dim * proj_dim)

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


HBP feature shape: torch.Size([8, 12288])
Expected dim: 12288


In [13]:
@torch.no_grad()
def extract_features(model, loader, device):
    model.eval()

    all_features = []
    all_labels = []
    all_image_ids = []

    for images, labels, image_ids in tqdm(loader, desc="Extracting HBP features"):
        images = images.to(device)

        feats = model(images)
        feats = feats.detach().cpu().numpy()

        all_features.append(feats)
        all_labels.extend(labels.numpy())
        all_image_ids.extend(list(image_ids))

    X = np.concatenate(all_features, axis=0)
    y = np.array(all_labels)

    return X, y, all_image_ids

In [14]:
X_train, y_train, train_image_ids = extract_features(hbp_model, train_loader, device)
X_val, y_val, val_image_ids = extract_features(hbp_model, val_loader, device)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

np.save(OUTPUT_DIR / "X_train_hbp.npy", X_train)
np.save(OUTPUT_DIR / "y_train.npy", y_train)
np.save(OUTPUT_DIR / "X_val_hbp.npy", X_val)
np.save(OUTPUT_DIR / "y_val.npy", y_val)

pd.DataFrame({"image_id": train_image_ids, "label": y_train}).to_csv(
    OUTPUT_DIR / "train_feature_ids.csv",
    index=False
)

pd.DataFrame({"image_id": val_image_ids, "label": y_val}).to_csv(
    OUTPUT_DIR / "val_feature_ids.csv",
    index=False
)

Extracting HBP features: 100%|██████████| 126/126 [00:20<00:00,  6.00it/s]


X_train: (9013, 12288)
y_train: (9013,)
X_val: (1002, 12288)
y_val: (1002,)


In [15]:
print("train feature mean:", X_train.mean())
print("train feature std:", X_train.std())
print("train feature min:", X_train.min())
print("train feature max:", X_train.max())

print("Any NaN:", np.isnan(X_train).any(), np.isnan(X_val).any())
print("Any Inf:", np.isinf(X_train).any(), np.isinf(X_val).any())

train feature mean: 2.7662274e-05
train feature std: 0.009021065
train feature min: -0.057330195
train feature max: 0.06050121
Any NaN: False False
Any Inf: False False


In [16]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("scaled train:", X_train_scaled.shape)

scaled train: (9013, 12288)


In [17]:
pca = PCA(
    n_components=pca_dim,
    random_state=seed
)

X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)

print("X_train_pca:", X_train_pca.shape)
print("X_val_pca:", X_val_pca.shape)
print("explained variance ratio sum:", pca.explained_variance_ratio_.sum())

with open(OUTPUT_DIR / "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open(OUTPUT_DIR / "pca.pkl", "wb") as f:
    pickle.dump(pca, f)

np.save(OUTPUT_DIR / "X_train_pca.npy", X_train_pca)
np.save(OUTPUT_DIR / "X_val_pca.npy", X_val_pca)

X_train_pca: (9013, 512)
X_val_pca: (1002, 512)
explained variance ratio sum: 0.90730023


In [18]:
rf = RandomForestClassifier(
    n_estimators=n_estimators,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=seed,
    n_jobs=-1,
    verbose=1
)

rf.fit(X_train_pca, y_train)

with open(OUTPUT_DIR / "random_forest.pkl", "wb") as f:
    pickle.dump(rf, f)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 160 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:    3.9s finished


In [19]:
y_pred = rf.predict(X_val_pca)
y_prob = rf.predict_proba(X_val_pca)

print(y_pred.shape)
print(y_prob.shape)

(1002,)
(1002, 7)


[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done 160 tasks      | elapsed:    0.1s
[Parallel(n_jobs=20)]: Done 300 out of 300 | elapsed:    0.1s finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done 160 tasks      | elapsed:    0.1s
[Parallel(n_jobs=20)]: Done 300 out of 300 | elapsed:    0.1s finished


In [20]:
metrics = {
    "accuracy": accuracy_score(y_val, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_val, y_pred),
    "precision_macro": precision_score(y_val, y_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_val, y_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_val, y_pred, average="macro", zero_division=0),
}

metrics

{'accuracy': 0.6696606786427146,
 'balanced_accuracy': np.float64(0.14285714285714285),
 'precision_macro': 0.09566581123467352,
 'recall_macro': 0.14285714285714285,
 'f1_macro': 0.11459311758176073}

In [21]:
pd.DataFrame([metrics]).to_csv(
    OUTPUT_DIR / "metrics.csv",
    index=False
)

print(pd.DataFrame([metrics]))

   accuracy  balanced_accuracy  precision_macro  recall_macro  f1_macro
0  0.669661           0.142857         0.095666      0.142857  0.114593


In [22]:
from collections import Counter

print("y_val distribution:")
print(Counter(y_val))

print("\ny_pred distribution:")
print(Counter(y_pred))

print("\nlabel mapping:")
print(idx2label)

y_val distribution:
Counter({np.int64(5): 671, np.int64(4): 111, np.int64(2): 110, np.int64(1): 51, np.int64(0): 33, np.int64(6): 14, np.int64(3): 12})

y_pred distribution:
Counter({np.int64(5): 1002})

label mapping:
{0: 'akiec', 1: 'bcc', 2: 'bkl', 3: 'df', 4: 'mel', 5: 'nv', 6: 'vasc'}
